# Day 1-3 — Exploratory Data Analysis: UCI Credit Card Default

**Dataset:** UCI Default of Credit Card Clients  
**Target:** `default` (1 = default next month, 0 = no default)

## Sections
1. Load Data
2. Basic Info & Descriptive Statistics
3. Target Distribution (Class Imbalance)
4. Demographic Features
5. Payment History Features
6. Bill & Payment Amount Features
7. Correlation Analysis
8. Key Observations & Next Steps

## 1. Load Data

In [ ]:
import sys
from pathlib import Path

# Add project root to path so we can import from src/
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_ingestion import load_data

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
# Downloads if needed, cleans, saves to parquet, returns DataFrame
df = load_data(
    raw_path=PROJECT_ROOT / 'data/raw/UCI_Credit_Card.xls',
    processed_path=PROJECT_ROOT / 'data/processed/credit_default_cleaned.parquet'
)
print("Data loaded successfully.")
print(f"Shape: {df.shape}")

## 2. Basic Info & Descriptive Statistics

In [ ]:
print("Dataset Info:")
df.info()

In [ ]:
print("Descriptive Statistics:")
df.describe()

In [ ]:
print("Missing Values:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "None — clean dataset!")

## 3. Target Distribution — Class Imbalance

> **Interview talking point:** ~22% default rate means accuracy is a misleading metric. Always report AUC-ROC, Gini, and KS statistic — these are threshold-independent.

In [ ]:
target_counts = df['default'].value_counts()
target_pct    = df['default'].value_counts(normalize=True) * 100

print("Target variable distribution:")
print(pd.DataFrame({'Count': target_counts, 'Percent': target_pct.round(1)}))

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['No Default (0)', 'Default (1)'], target_counts, color=['#2196F3', '#F44336'])
for bar, pct in zip(bars, target_pct):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{pct:.1f}%', ha='center', fontweight='bold')
ax.set_title('Distribution of Default Payment Next Month')
ax.set_ylabel('Count')
ax.set_xlabel('Default (1) / No Default (0)')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/01_target_distribution.png', dpi=150)
plt.show()

## 4. Demographic Features

**Codebook:**
- `SEX`: 1=male, 2=female
- `EDUCATION`: 1=graduate school, 2=university, 3=high school, 4=others
- `MARRIAGE`: 1=married, 2=single, 3=others

In [ ]:
# LIMIT_BAL distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['LIMIT_BAL'], bins=50, kde=True, ax=axes[0])
axes[0].set_title('Distribution of LIMIT_BAL')
axes[0].set_xlabel('Credit Limit (NT dollar)')
axes[0].set_ylabel('Frequency')

sns.boxplot(x='default', y='LIMIT_BAL', data=df, ax=axes[1])
axes[1].set_title('LIMIT_BAL vs. Default Status')
axes[1].set_xlabel('Default (1) / No Default (0)')
axes[1].set_ylabel('Credit Limit (NT dollar)')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/02_limit_bal.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Default rate by SEX
sex_default = df.groupby('SEX')['default'].mean() * 100
sex_default.index = ['Male', 'Female']
sex_default.plot(kind='bar', ax=axes[0,0], color='#FF9800', rot=0)
axes[0,0].set_title('Default Rate by Gender (%)')
axes[0,0].set_ylabel('Default Rate (%)')

# Default rate by EDUCATION
edu_default = df.groupby('EDUCATION')['default'].mean() * 100
edu_default.index = ['Grad School', 'University', 'High School', 'Others']
edu_default.plot(kind='bar', ax=axes[0,1], color='#9C27B0', rot=30)
axes[0,1].set_title('Default Rate by Education (%)')

# Default rate by MARRIAGE
mar_default = df.groupby('MARRIAGE')['default'].mean() * 100
mar_default.index = ['Married', 'Single', 'Others']
mar_default.plot(kind='bar', ax=axes[0,2], color='#4CAF50', rot=0)
axes[0,2].set_title('Default Rate by Marital Status (%)')

# AGE distribution by default
df[df.default==0]['AGE'].plot(kind='hist', bins=30, ax=axes[1,0], alpha=0.6, label='No Default', color='#2196F3')
df[df.default==1]['AGE'].plot(kind='hist', bins=30, ax=axes[1,0], alpha=0.6, label='Default', color='#F44336')
axes[1,0].set_title('Age Distribution by Default Status')
axes[1,0].legend()

# Default rate by age group
df['age_group'] = pd.cut(df['AGE'], bins=[20,30,40,50,60,80], labels=['20s','30s','40s','50s','60+'])
age_default = df.groupby('age_group', observed=True)['default'].mean() * 100
age_default.plot(kind='bar', ax=axes[1,1], color='#00BCD4', rot=0)
axes[1,1].set_title('Default Rate by Age Group (%)')

# Credit limit KDE by default
df.groupby('default')['LIMIT_BAL'].plot(kind='kde', ax=axes[1,2], legend=True)
axes[1,2].set_title('Credit Limit KDE by Default Status')
axes[1,2].set_xlabel('Credit Limit')

plt.suptitle('Demographic Features vs Default', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/03_demographic_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Payment History Features (PAY_0 through PAY_6)

**Codebook:** -1=paid duly, 0=minimum paid, 1=one month delay, 2=two months delay, ...

> **Key insight:** Repayment status in recent months is the strongest predictor of default.

In [ ]:
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
months   = ['Sep 2005', 'Aug 2005', 'Jul 2005', 'Jun 2005', 'May 2005', 'Apr 2005']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, (col, month) in enumerate(zip(pay_cols, months)):
    pay_default = df.groupby(col)['default'].mean() * 100
    pay_default.plot(kind='bar', ax=axes[i], color='#FF5722', rot=0)
    axes[i].set_title(f'{col} ({month}) — Default Rate')
    axes[i].set_xlabel('Payment Status')
    axes[i].set_ylabel('Default Rate (%)')

plt.suptitle('Payment Status vs Default Rate (by Month)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/04_payment_history.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nCorrelation with default:')
print(df[pay_cols + ['default']].corr()['default'].sort_values(ascending=False))

## 6. Bill & Payment Amount Features

In [ ]:
bill_cols    = ['BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6']
pay_amt_cols = ['PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('default')[bill_cols].median().T.plot(ax=axes[0])
axes[0].set_title('Median Bill Amount by Month & Default Status')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Median Bill (NTD)')
axes[0].legend(['No Default', 'Default'])

df.groupby('default')[pay_amt_cols].median().T.plot(ax=axes[1])
axes[1].set_title('Median Payment Amount by Month & Default Status')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Median Payment (NTD)')
axes[1].legend(['No Default', 'Default'])

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/05_bill_payment_amounts.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Correlation Analysis

In [ ]:
corr_with_default = (
    df.select_dtypes(include='number')
      .corr()['default']
      .drop('default')
      .abs()
      .sort_values(ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

corr_with_default.head(15).plot(kind='barh', ax=axes[0], color='#3F51B5')
axes[0].set_title('Top 15 Features by |Correlation| with Default')
axes[0].set_xlabel('|Correlation|')
axes[0].invert_yaxis()

key_cols = pay_cols + ['LIMIT_BAL', 'AGE', 'default']
sns.heatmap(df[key_cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=axes[1], linewidths=0.5)
axes[1].set_title('Correlation Heatmap — Payment History + Key Features')

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'reports/figures/06_correlation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Top 10 features correlated with default:')
print(corr_with_default.head(10))

## 8. Key Observations & Next Steps

| Observation | Implication |
|-------------|-------------|
| ~22% default rate | Class imbalance — use `class_weight='balanced'` or SMOTE |
| PAY_0 strongest predictor | Recent payment behavior is the #1 signal |
| Defaulters pay significantly less | Payment ratio feature will be highly predictive |
| EDUCATION / MARRIAGE undocumented categories | Already binned in `data_ingestion.clean()` |
| No missing values | Clean dataset — focus energy on feature engineering |

### Day 4 → Feature Engineering Ideas
```python
# 1. Payment ratio: PAY_AMT / BILL_AMT (what fraction of bill was paid)
# 2. Credit utilization: BILL_AMT1 / LIMIT_BAL
# 3. Delinquency streak: consecutive months with PAY > 0
# 4. Max delinquency: max(PAY_0..PAY_6)
# 5. Payment trend: is payment amount increasing or decreasing?
```

### Interview Talking Points
- **On class imbalance:** "A naive model predicting 'no default' gets 78% accuracy — that's why I use AUC-ROC and Gini, which are threshold-independent."
- **On PAY_0 dominance:** "Recent payment behavior dominates because it directly signals current financial stress — aligns with behavioral scoring models used in industry."
- **On data cleaning:** "Even a 'clean' dataset had undocumented categories in EDUCATION and MARRIAGE — binned in the ingestion layer so every downstream consumer gets consistent data."

In [ ]:
# Drop temp column and save final base dataset
df = df.drop(columns=['age_group'], errors='ignore')
df.to_parquet(PROJECT_ROOT / 'data/processed/credit_default_cleaned.parquet', index=True)
print(f'Saved base dataset. Shape: {df.shape}')